# Cheminformatics Eval — execute agent, **with vs without skills**

This notebook evaluate if the **cheminformatics** skill helps the ChemSafe's **execute agent** answers
cheminformatics questions better.

## Method
Given the same set of questions, the **execute agennt** attempts to answer those with and without **cheminformatics** skill. 

Both conditions can still call libraries like `rdkit`, `admet_ai`, `deepchem`, and `pubchempy` from Python; 
the only thing vary is the incorporation of **cheminforatics skill**. So the difference in scores isolates the value of the skills.

### LLM-as-judge scorer

Each agent answer is free-form text. A second model grades it and returns
`correct` / `partial` / `incorrect`, matching numbers within **±2% relative**
tolerance (±1e-3 near zero), categorical labels (GHS class, H-codes,
active/inactive, …) by meaning, SMILES by chemical equivalence, and prose by
whether the key claims are present. (See more details in section 7)

## 1. Import dependencies

In [ ]:
import csv
import glob
import json
import os
import random
import re
import sys
import time
import traceback
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Literal, Optional, Union
import pandas as pd  


def _find_repo_root(start: Path) -> Path:
    """Walk up from `start` until we find the project, and return that folder."""
    for parent in [start, *start.parents]:
        if (parent / "core" / "agents" / "execute_agent.py").exists():
            return parent
    raise RuntimeError(
        "Could not find the repo root. Open this notebook from inside the "
        "chemsafe-agent repository."
    )


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root:", REPO_ROOT)

repo root: /Users/dinhu955/Desktop/ChemSafeAgent/chemsafe-agent
eval dir : /Users/dinhu955/Desktop/ChemSafeAgent/chemsafe-agent/analysis/cheminformatics_eval


In [58]:
from langchain.chat_models import init_chat_model           
from langchain_core.messages import AIMessage, HumanMessage  
from langgraph.prebuilt import create_react_agent            
from pydantic import BaseModel, Field                        


from app.config import EXECUTE_MODEL, OPENAI_API_KEY, SUMMARY_MODEL  
from core.agents.execute_agent import build_execute_agent            
from core.agents.context import build_uncompressed_pre_model_state   
from core.prompts.prompts import EXECUTE_AGENT_FREE_SYSTEM_PROMPT, EXECUTE_SKILLS_BLOCK
from backend.utils.skills_format import format_skill_summaries

from core.tools.python_executor import python_executor, reset_python_state  
from backend.utils.output_paths import (                           
    set_current_conversation_id,
    set_current_user_id,
)

## 2. Pin an output / read scope

In [59]:
EVAL_USER_ID = "cheminf-eval"
EVAL_CONVERSATION_ID = "cheminf-eval-run"


def set_eval_scope(user_id: str = EVAL_USER_ID, conversation_id: str = EVAL_CONVERSATION_ID):
    """Pin the output/read scope for the eval session (idempotent)."""
    set_current_user_id(user_id)
    set_current_conversation_id(conversation_id)

## 3. Build the WITH-skills agent

This builds the **execute agent** with **cheminformatics** skill

In [ ]:
EVAL_SKILLS = ["cheminformatics", "database_traversal"]
EVAL_SKILLS_BLOCK = format_skill_summaries(EVAL_SKILLS)
# Only activate two skills rather than full skill set of execution agent
EVAL_EXECUTE_SYSTEM_PROMPT = EXECUTE_AGENT_FREE_SYSTEM_PROMPT.replace(
    EXECUTE_SKILLS_BLOCK, EVAL_SKILLS_BLOCK
)

def build_eval_agent(model: Optional[str] = None):
    """Build the execute agent restricted to the EVAL_SKILLS (cheminformatics +
    database_traversal).
    """
    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY is not set. Add it to the repo-root .env.")
    llm = init_chat_model(model or EXECUTE_MODEL, model_provider="openai", api_key=OPENAI_API_KEY)
    prompt, name = EVAL_EXECUTE_SYSTEM_PROMPT, "execute_agent_free"
    return build_execute_agent(
        llm, pre_model_hook=build_uncompressed_pre_model_state, name=name, prompt=prompt
    )

## 4. Build the NO-skills baseline agent

This builds the **execute agent** without **cheminformatics** skill

In [61]:
NO_SKILLS_SYSTEM_PROMPT = """
You are the execution agent for chemical safety-relevant workflows. You handle short, well-scoped requests directly, without an external plan. Your job is to reach a correct, evidence-grounded answer efficiently and use tools whenever they produce real progress.

--- 

## PRIMARY ROLE

1. Keep the user's request as the source of truth.
2. Fully solve the request. Treat every task as potentially non-trivial: consider the real scope before deciding how much work it needs, and
   do not assume a request is small just because it is short.
3. Use tools to produce real evidence (file contents, executions,
   validations, SOP-grounded answers). When concrete data could change the
   answer, get it. Do not answer from reasoning alone.
4. Match your effort to the actual difficulty of the task. When the scope is
   ambiguous, err toward doing more verification and more thorough work
   rather than less.
5. Surface any uncertainty or limitation explicitly instead of hiding it.

You must not:
- Underestimate a task or stop early. If parts of the request remain
  unaddressed or unverified, the work is not done.
- Skip tool use and answer purely from reasoning when concrete files, data,
  SOPs, or executions are needed.

---

## Execution Posture

The expected runtime pattern is:

1. Restate the request in one short line and identify what evidence the answer needs.
2. Load only the context required.
3. Execute with one or more focused tool calls.
4. Inspect results and either finish or recover from errors.
5. Produce the final answer grounded in observed evidence.

Prefer small, focused `python_executor` probes. Reuse Python state when useful; `reset_python_state` when it gets stale.

---

## Tool Discipline

- `python_executor` performs inspection, lightweight analysis, validation, and file generation.
- `reset_python_state` is a recovery tool for contaminated Python state.

**Rules:**

- Do not declare the task complete from reasoning alone when tool evidence is available.
- Recovery is part of execution — adapt the approach on failure rather than abandoning.
- Use the injected output helpers (`prepare_output_path`, `ensure_output_dir`) for any generated files.

---

## Safety Guardrails

1. For any safety-relevant action, threshold, or recommendation, ground it in computed or cited evidence before finalizing the answer.
2. For any data-dependent claim, inspect the data before summarizing it.
3. Never invent tool outputs, values, thresholds, or results.
4. If you cannot complete the request safely, say so explicitly and surface the blocker.
"""

def build_eval_agent_no_skills(model: Optional[str] = None):
    """Baseline agent: python_executor only (NO read_files), skill-free prompt."""
    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY is not set. Add it to the repo-root .env.")
    llm = init_chat_model(model or EXECUTE_MODEL, model_provider="openai", api_key=OPENAI_API_KEY)
    return create_react_agent(
        model=llm,
        tools=[python_executor, reset_python_state],  # no read_files -> no skill to read
        name="execute_agent_no_skills",
        prompt=NO_SKILLS_SYSTEM_PROMPT,
        pre_model_hook=build_uncompressed_pre_model_state,
    )

## 5. Load the evaluation dataset

Loads the CSV into a list of plain dictionaries (one per question). Two cleaning
steps matter:

* The CSV's column headers are wrapped in single quotes, so `_clean_key` strips them.
* For `dict` / `scalar` rows the reference answer also ships as JSON in the
  `ref_answer_parsed` column; `_safe_json` parses it into `row['_ref_parsed']` (shown to
  the judge as the structured reference). `prose` rows have none.

In [ ]:
def _clean_key(key: str) -> str:
    """Strip surrounding quotes/spaces from a CSV column name."""
    return (key or "").strip().strip("'\"").strip()


def _safe_json(text: str):
    """Parse a JSON string, or return None if it is empty/invalid."""
    text = (text or "").strip()
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        return None


def load_dataset(path: Optional[Union[str, Path]] = None) -> list[dict]:
    """Load the eval CSV into normalized row dicts.

    Adds row['_ref_parsed']: the parsed JSON reference for dict/scalar rows.
    """
    p = Path(path)
    with open(p, newline="", encoding="utf-8") as fh:
        raw_rows = list(csv.DictReader(fh))
    rows: list[dict] = []
    for raw in raw_rows:
        row = {_clean_key(k): (v if v is not None else "") for k, v in raw.items()}
        row["_ref_parsed"] = _safe_json(row.get("ref_answer_parsed", ""))
        rows.append(row)
    return rows

## 6. Run the agent on one question (and capture a trace)

`run_agent_on_query` sends one question to an agent and collects the result. The agent returns a list of chat messages:

* `_text` flattens a message's content to plain text.
* `_final_answer` picks the **last assistant message that is real text** (not a tool
  call) — that's the agent's answer.
* `_summarize_trace` inspects the messages to record **how** the agent worked: which
  tools it called, how many Python runs, which `SKILL.md` files it opened, and whether
  its code imported `core.skills`. (This is how we can later confirm the no-skills
  agent really used no skills.)

Any error is caught and returned as `ok=False`, so a single bad row never aborts a long
run.

In [ ]:
def _text(content: Any) -> str:
    """Flatten a message's content (str, or list of parts) into plain text."""
    if content is None:
        return ""
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                parts.append(item.get("text", ""))
            elif isinstance(item, str):
                parts.append(item)
        return "\n".join(parts).strip()
    return str(content).strip()


def _final_answer(messages: list) -> str:
    """Last assistant message with text content and no pending tool calls."""
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and not getattr(msg, "tool_calls", None):
            text = _text(msg.content)
            if text:
                return text
    for msg in reversed(messages):  # fallback
        if isinstance(msg, AIMessage):
            text = _text(msg.content)
            if text:
                return text
    return ""


def _summarize_trace(messages: list) -> dict:
    """Record how the agent worked: tools used, SKILL.md files read, core.skills imports."""
    tools_used: list[str] = []
    skills_read: set[str] = set()
    skills_in_code: set[str] = set()
    n_python_runs = 0
    for msg in messages:
        for call in getattr(msg, "tool_calls", None) or []:
            name = call.get("name", "")
            args = call.get("args", {}) or {}
            tools_used.append(name)
            if name == "read_files":
                fp = str(args.get("file_path", ""))
                if "skills/" in fp:
                    skills_read.add(fp.split("skills/")[1].split("/")[0])
            elif name == "python_executor":
                n_python_runs += 1
                for mod in re.findall(r"core\.skills\.(\w+)", str(args.get("code", ""))):
                    skills_in_code.add(mod)
    return {
        "tool_counts": dict(Counter(tools_used)),
        "n_python_runs": n_python_runs,
        "skills_read": sorted(skills_read),
        "skills_used_in_code": sorted(skills_in_code),
    }

  

def run_agent_on_query(agent, question: str, *, recursion_limit: int = 100) -> dict:
    """Invoke the agent on one question; return answer + trace + timing.
    Retries agent.invoke up to 5 times, returning on the first success.
    recursion_limit: max agent steps (the agent may take many tool turns).
    """
    started = time.time()
    last_error, last_tb = "", ""

    for retry in range(10):
        try:
            result = agent.invoke(
                {"messages": [HumanMessage(content=question)]},
                config={"recursion_limit": recursion_limit},
            )
            messages = result["messages"]
            return {
                "ok": True,
                "answer": _final_answer(messages),
                "trace": _summarize_trace(messages),
                "n_messages": len(messages),
                "elapsed_s": round(time.time() - started, 2),
            }
        except (Exception, SystemExit) as exc:  # record and try again
            last_error = f"{type(exc).__name__}: {exc}"
            last_tb = traceback.format_exc()

    # all attempts failed
    return {
        "ok": False,
        "answer": "",
        "error": last_error,
        "traceback": last_tb,
        "elapsed_s": round(time.time() - started, 2),
    }

## 7. The LLM judge

The verdict comes from a second LLM acting as a grader.

* `JudgeVerdict` is the **Pydantic model** that is forced the judge to return
  (`correct`/`partial`/`incorrect` plus supporting detail). Using structured output
  means we get clean fields back, not free text to parse.
* `JUDGE_SYSTEM` is the rubric: numbers within tolerance, categorical labels matched by
  meaning, SMILES by chemical equivalence, prose by claim coverage, and "don't penalize
  extra correct content".
* `build_judge` wires up the model with structured output.
* `judge_answer` hands the judge the question, the reference (and its parsed form), and
  the agent's answer, and returns the verdict as a dict.

In [ ]:
class JudgeVerdict(BaseModel):
    """Structured verdict the judge must return.

    Field order is deliberate: the judge fills in its reasoning and field-level
    evidence FIRST and commits to `verdict` LAST, so the verdict is conditioned
    on the reasoning rather than the reverse (CoT-first / form-filling, cf.
    G-Eval; MT-Bench's "explain first, then rate").
    """

    reasoning: str = Field(
        description="Grade step by step: (1) state what a fully correct answer "
        "must contain given the reference, (2) compare the agent answer field by "
        "field, (3) only then decide. 1-4 sentences."
    )
    matched_fields: list[str] = Field(default_factory=list)
    missing_or_wrong_fields: list[str] = Field(default_factory=list)
    numeric_within_tolerance: Optional[bool] = Field(
        default=None,
        description="True/False if the reference contains numeric values; null "
        "if the answer is purely categorical/textual.",
    )
    verdict: Literal["correct", "partial", "incorrect"] = Field(
        description="correct = all required content right; partial = primary "
        "value right but secondary fields wrong/missing; incorrect = primary "
        "requested value wrong or absent."
    )


JUDGE_SYSTEM = """You are a strict, fair, REFERENCE-GUIDED evaluator for a
cheminformatics agent. You are given a question, a reference answer (and, when
available, a structured parsed form of it), and the agent's free-text answer.

HOW TO GRADE (do this in `reasoning` BEFORE choosing `verdict`):
1. From the reference (and reference_parsed), state what a fully correct answer
   must contain.
2. Compare the agent answer to that requirement, field by field; populate
   matched_fields vs missing_or_wrong_fields.
3. Only then commit to a verdict.

SCORING RULES
- Numeric values: a match is within +/-2% relative tolerance (or +/-1e-3
  absolute when the reference is ~0). The agent uses the same underlying
  RDKit/QSAR models as the reference, so correct values should be very close; a
  large deviation is WRONG, not a rounding difference.
- Categorical values (GHS hazard class, H-codes, signal word DANGER/WARNING,
  Tox21 active/inactive, AD inside/outside, alert yes/no): must match in
  meaning, ignoring formatting/case/phrasing differences.
- SMILES: correct when they denote the same molecule (canonical equivalence);
  ignore cosmetic formatting differences.

BIAS CONTROLS (apply strictly)
- Do NOT reward length, verbosity, or confidence. A longer or more assertive
  answer is not a better answer; grade only correctness against the reference.
- Do NOT penalize extra correct content (caveats, AD/units context, extra
  correct reasoning). Penalize only content that is wrong, contradictory, or
  required-but-missing.
- Grade the chemistry, not the writing style.

VERDICT BY answer_type
- "dict": reference_parsed lists the required fields/keys. correct = every
  required field conveyed correctly; partial = the primary requested quantity
  is right but a secondary field is wrong/missing; incorrect = the primary
  requested value is wrong or absent.
- "scalar": reference_parsed is {"value": ...}; correct iff that single value
  matches, else incorrect.
- "prose": reference_parsed is null; judge whether the agent answer conveys the
  same key conclusions/claims as the reference. correct = all key claims present
  and none contradicted; partial = some; incorrect = main conclusion wrong/absent.
"""


def build_judge(model: Optional[str] = "gpt-5.4", temperature: float = 0.0):
    """LLM judge with structured output.
    """
    base = init_chat_model(
        model, model_provider="openai", api_key=OPENAI_API_KEY, temperature=temperature
    )
    return base.with_structured_output(JudgeVerdict)


def _judge_payload(row: dict, agent_answer: str) -> str:
    return JUDGE_SYSTEM + "\n\nEVALUATION INPUT (JSON):\n" + json.dumps(
        {
            "question": row.get("question", ""),
            "answer_type": row.get("answer_type", ""),
            "reference_answer": row.get("ref_answer", ""),
            "reference_parsed": row.get("_ref_parsed"),
            "agent_answer": agent_answer,
        },
        ensure_ascii=False,
        indent=2,
    )


def judge_answer(judge, row: dict, agent_answer: str, *, max_attempts: int = 5) -> dict:
    """One verdict from the judge; retries with exponential backoff."""
    message = _judge_payload(row, agent_answer)
    last_exc = None
    for attempt in range(max_attempts):
        try:
            return judge.invoke(message).model_dump()
        except Exception as exc:
            last_exc = exc
            time.sleep(min(2 ** attempt, 8))
    return {
        "reasoning": f"judge failed after {max_attempts} attempts: "
        f"{type(last_exc).__name__}: {last_exc}",
        "matched_fields": [],
        "missing_or_wrong_fields": [],
        "numeric_within_tolerance": None,
        "verdict": "error",
    }



## 8. Score one row (run + judge)

`score_row` is the unit of work for a single question: run the agent, then ask the
judge (unless the agent crashed). It returns **one flat record** — inputs, the agent's
answer, the judge verdict, the trace, and the `condition` tag — which is exactly what
gets saved to disk.

In [65]:
def score_row(agent, judge, row: dict, *, condition: str = "with_skills", **run_kwargs) -> dict:
    """Run the agent on one row and score it; return a single result record."""

    run = run_agent_on_query(agent, row.get("question", ""), **run_kwargs)
    
    answer = run.get("answer", "")
    if run["ok"]:
        verdict = judge_answer(judge, row, answer)
    else:
        verdict = {
            "verdict": "incorrect",
            "numeric_within_tolerance": None,
            "matched_fields": [],
            "missing_or_wrong_fields": [],
            "rationale": f"agent run failed: {run.get('error')}",
        }
    return {
        "condition": condition,
        "question_id": row.get("question_id"),
        "domain": row.get("domain"),
        "difficulty": row.get("difficulty"),
        "answer_type": row.get("answer_type"),
        "question": row.get("question"),
        "ref_answer": row.get("ref_answer"),
        "ref_answer_parsed": row.get("ref_answer_parsed"),
        "agent_answer": answer,
        "agent_ok": run["ok"],
        "error": run.get("error"),
        "elapsed_s": run.get("elapsed_s"),
        "trace": run.get("trace"),
        "judge": verdict,
        "verdict": verdict.get("verdict"),
    }

## 9. Save results so runs are resumable

Results are written as **JSON Lines** (one JSON object per line) and **appended**, so a
run can be stopped and resumed without losing work. `load_results` reads them back.
`done_keys` returns the `(condition, question_id)` pairs already finished successfully —
the orchestrator uses it to skip work that's already done.

In [ ]:
# This folder holds the notebook, the dataset CSV, and the results we write.
EVAL_DIR = REPO_ROOT / "analysis" / "cheminformatics_eval"
RESULTS_PATH = EVAL_DIR / "eval_runs" / "results.jsonl"     


def append_result(record: dict, path: Union[str, Path] = RESULTS_PATH) -> None:
    """Append one result record as a line of JSON."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def load_results(path: Union[str, Path] = RESULTS_PATH) -> list[dict]:
    """Read back all result records from a JSONL file (empty list if none)."""
    path = Path(path)
    if not path.exists():
        return []
    out = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                try:
                    out.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return out


def done_ids(path: Union[str, Path] = RESULTS_PATH) -> set:
    """question_ids that already have a successful run (any condition)."""
    return {str(r.get("question_id")) for r in load_results(path) if r.get("agent_ok")}


def done_keys(path: Union[str, Path] = RESULTS_PATH) -> set:
    """(condition, question_id) pairs already finished successfully — the resume key."""
    return {
        (str(r.get("condition", "with_skills")), str(r.get("question_id")))
        for r in load_results(path)
        if r.get("agent_ok")
    }

## 10. Orchestrate a run over many rows

`run_eval` loops over rows: skip anything already done (resume), score each remaining
row, append the result, and print progress. The `condition` string tags every record
and is part of the resume key, so both conditions can safely share one results file.

`run_with_skills` / `run_no_skills` are thin wrappers that select the right agent and
condition for you (the agent's own system prompt is what makes it "with" or "no"
skills — there is nothing else to pass).

In [67]:
def run_eval(
    rows: list[dict],
    agent=None,
    judge=None,
    *,
    condition: str = "with_skills",
    limit: Optional[int] = None,
    resume: bool = True,
    path: Union[str, Path] = RESULTS_PATH,
    verbose: bool = True,
    **run_kwargs,
) -> list[dict]:
    """Run the agent + judge over `rows`, appending each result to JSONL (resumable)."""
    agent = agent or build_eval_agent()
    judge = judge or build_judge(model = 'gpt-5.4')
    set_eval_scope()

    skip = done_keys(path) if resume else set()
    todo = [r for r in rows if (condition, str(r.get("question_id"))) not in skip]
    if limit is not None:
        todo = todo[:limit]

    if verbose:
        print(
            f"[{condition}] running {len(todo)} rows "
            f"(skipping {sum(1 for k in skip if k[0] == condition)} already done). -> {path}"
        )

    out: list[dict] = []
    for i, row in enumerate(todo, 1):
        record = score_row(agent, judge, row, condition=condition, **run_kwargs)
        append_result(record, path)
        out.append(record)
        if verbose:
            print(
                f"[{condition}][{i}/{len(todo)}] qid={record['question_id']} "
                f"{record['domain']}/{record['answer_type']} -> "
                f"{record['verdict']} ({record['elapsed_s']}s)"
            )
    return out


def run_with_skills(rows, agent=None, judge=None, **kwargs):
    """Run the with-skills condition (real execute agent: has read_files + skill prompt)."""
    return run_eval(
        rows,
        agent=agent or build_eval_agent(),
        judge=judge,
        condition="with_skills",
        **kwargs,
    )


def run_no_skills(rows, agent=None, judge=None, **kwargs):
    """Run the no-skills baseline (python_executor only + skill-free prompt)."""
    return run_eval(
        rows,
        agent=agent or build_eval_agent_no_skills(),
        judge=judge,
        condition="no_skills",
        **kwargs,
    )

## 11. Instantiate — build both agents, the judge, and load the data

Everything above only *defined* things. The cell below actually constructs the two
agents and the judge, and loads the dataset. (Constructing the agents does not call the
API; only running them in the next sections does.)

In [68]:
set_eval_scope()

agent_skills    = build_eval_agent()   # with skills (has read_files)
agent_no_skills = build_eval_agent_no_skills()        # python_executor only
judge           = build_judge(model = 'gpt-5.4')
rows            = load_dataset('cheminformatics_eval_dataset.csv')

## 12. Run the full evaluation

In [ ]:
# --- FULL RUN, both conditions (uncomment; resumable). ---
run_with_skills(rows, agent=agent_skills, judge=judge)
run_no_skills(rows, agent=agent_no_skills, judge=judge)

## 13. Compare: with skills vs no skills

`compare_conditions` reads the shared results file and shows overall accuracy per
condition, per-domain accuracy with the delta, the paired per-question view, and the
`helped` / `hurt` lists. If the full run hasn't been done yet, it falls back to the
smoke comparison so the cell still demonstrates the output.

In [ ]:
import json
import pandas as pd

PATH = "eval_runs/results.jsonl"  # adjust to your file location


def load_jsonl(path):
    dec = json.JSONDecoder()
    text = open(path, encoding="utf-8").read()
    i, n = 0, len(text)
    while i < n:
        while i < n and text[i] in " \r\n\t":   # skip whitespace/newlines
            i += 1
        if i >= n:
            break
        obj, end = dec.raw_decode(text, i)
        i = end
        yield obj

raw = pd.DataFrame(load_jsonl(PATH))

SCORE_MAP = {"correct": 1.0, "partial": 0.5, "incorrect": 0.0}

# --- Long-format table with the requested columns ---
df = pd.DataFrame({
    "question_type": raw["domain"],                       # swap to raw["answer_type"] if you meant dict/prose/scalar
    "difficulty":    raw["difficulty"],
    "skills":        raw["condition"].eq("with_skills"),   # True / False
    "accuracy":      raw["verdict"],                       # correct / incorrect / partial
})
df["score"] = df["accuracy"].map(SCORE_MAP)

# Optional: give difficulty and accuracy a natural order (helps groupby/plots)
df["difficulty"] = pd.Categorical(df["difficulty"], ["easy", "medium", "hard"], ordered=True)
df["accuracy"]   = pd.Categorical(df["accuracy"], ["incorrect", "partial", "correct"], ordered=True)